In [6]:
import pandas as pd

print("="*60)
print("👪 VELİ ANKETİ: MİN-MAX STANDARTLAŞTIRMALI İLGİ SKORU")
print("="*60)

# 1. Temizlenmiş veli verisini okuma
df_veliler = pd.read_csv("../../data/processed/veliler_temiz.csv")

# 2. Hiyerarşik Puanlama Sözlükleri
sozluk_denetim = {
    'Evet, teknik ayarları (filtre, süre sınırı) aktif olarak kullanıyorum.': 4,
    'Nasıl yapılacağını biliyorum ama uygulamıyorum.': 3,
    'Bu tür ayarların nasıl yapıldığını bilmiyorum.': 2,
    'Cihazlarda böyle özelliklerin olduğundan haberdar değilim.': 1
}

sozluk_haftalik_gun = {
    'Hiçbir zaman': 5,
    'Nadiren (Ayda 1-2 kez)': 4,
    'Bazen (Haftada 1-2 kez)': 3,
    'Sıklıkla (Haftada 3-4 kez)': 2,
    'Neredeyse her gün / Her gün': 1,
    'Bilmiyorum': 0
}

sozluk_gunluk_sure = {
    '1 saatten az': 4,
    '1-3 saat': 3,
    '3-5 saat': 2,
    '5 saatten fazla': 1,
    'Bilmiyorum': 0
}

sozluk_harcama_izin = {
    'Hiç harcama yapmaz': 4,
    '250 TL\'den az': 3,
    '250 - 1000 TL arası': 2,
    '1000 TL ve üzeri': 1,
    'Bilmiyorum': 0
}

sozluk_harcama_etki = {
    'Hiç harcamaz veya sadece izin verdiğimiz kadar harcar.': 4,
    'Bazen limitini aşar ama bütçemizi sarsmaz.': 3,
    'Bütçemizi zorlar, sürekli uyarmam gerekir.': 2,
    'Kontrol edemeyiz veya bizden gizli/izinsiz harcama yapar.': 1,
    'İlgilenmiyorum / Bilmiyorum': 0
}

sozluk_pegi = {
    'Evet her zaman': 3,
    'Bazen': 2,
    'Hayır kontrol etmiyorum': 1,
    'PEGI nedir bilmiyorum': 0
}

# 3. Sayısallaştırma (Boşlukları veya tanımsızları 0 kabul ediyoruz)
df_veliler['skor_denetim'] = df_veliler['veli_ebeveyn_denetimi'].map(sozluk_denetim).fillna(0)
df_veliler['skor_haftalik_gun'] = df_veliler['cocuk_haftalik_oyun_gunu'].map(sozluk_haftalik_gun).fillna(0)
df_veliler['skor_gunluk_sure'] = df_veliler['cocuk_gunluk_sure'].map(sozluk_gunluk_sure).fillna(0)
df_veliler['skor_harcama_izin'] = df_veliler['veli_izin_verilen_harcama'].map(sozluk_harcama_izin).fillna(0)
df_veliler['skor_harcama_etki'] = df_veliler['veli_harcama_guven_etkisi'].map(sozluk_harcama_etki).fillna(0)
df_veliler['skor_pegi'] = df_veliler['veli_pegi_kontrol'].map(sozluk_pegi).fillna(0)

# =====================================================================
# 4. SORU BAZLI MİN-MAX STANDARTLAŞTIRMASI (0.0 - 1.0 ARALIĞI)
# Formül: Değer / O sorunun alabileceği maksimum puan
# =====================================================================
df_veliler['skor_denetim'] = df_veliler['skor_denetim'] / 4.0
df_veliler['skor_haftalik_gun'] = df_veliler['skor_haftalik_gun'] / 5.0
df_veliler['skor_gunluk_sure'] = df_veliler['skor_gunluk_sure'] / 4.0
df_veliler['skor_harcama_izin'] = df_veliler['skor_harcama_izin'] / 4.0
df_veliler['skor_harcama_etki'] = df_veliler['skor_harcama_etki'] / 4.0
df_veliler['skor_pegi'] = df_veliler['skor_pegi'] / 3.0

# 5. Gür & Türel (2025) Faktör Yüklerinin Atanması
agirlik_denetim = 0.51
agirlik_haftalik_gun = 0.64
agirlik_gunluk_sure = 0.76
agirlik_harcama_izin = 0.61
agirlik_harcama_etki = 0.62
agirlik_pegi = 0.61

# 6. Ağırlıklı Ham Skoru Hesaplama
df_veliler['ham_ilgi_skoru'] = (
    (df_veliler['skor_denetim'] * agirlik_denetim) +
    (df_veliler['skor_haftalik_gun'] * agirlik_haftalik_gun) +
    (df_veliler['skor_gunluk_sure'] * agirlik_gunluk_sure) +
    (df_veliler['skor_harcama_izin'] * agirlik_harcama_izin) +
    (df_veliler['skor_harcama_etki'] * agirlik_harcama_etki) +
    (df_veliler['skor_pegi'] * agirlik_pegi)
)

# 7. Nihai 100 Üzerinden Ölçeklendirme (Kullanıcı Dostu Skor)
min_skor = df_veliler['ham_ilgi_skoru'].min()
max_skor = df_veliler['ham_ilgi_skoru'].max()

if max_skor > min_skor: 
    df_veliler['ebeveyn_ilgi_skoru_100'] = ((df_veliler['ham_ilgi_skoru'] - min_skor) / (max_skor - min_skor)) * 100
else:
    df_veliler['ebeveyn_ilgi_skoru_100'] = 100

df_veliler['ebeveyn_ilgi_skoru_100'] = df_veliler['ebeveyn_ilgi_skoru_100'].round(0).astype(int)

# 8. Geçici Sütunları Temizleme
sutunlari_sil = ['skor_denetim', 'skor_haftalik_gun', 'skor_gunluk_sure', 
                 'skor_harcama_izin', 'skor_harcama_etki', 'skor_pegi', 'ham_ilgi_skoru']
df_veliler = df_veliler.drop(columns=sutunlari_sil)

print("✅ Şık dengesizlikleri giderildi ve skorlama tamamlandı!\n")
display(df_veliler[['veli_ebeveyn_denetimi', 'cocuk_gunluk_sure', 'ebeveyn_ilgi_skoru_100']].head())

👪 VELİ ANKETİ: MİN-MAX STANDARTLAŞTIRMALI İLGİ SKORU
✅ Şık dengesizlikleri giderildi ve skorlama tamamlandı!



,veli_ebeveyn_denetimi,cocuk_gunluk_sure,ebeveyn_ilgi_skoru_100
0,"Evet, teknik ayarları (filtre, süre sınırı) ak...",1 saatten az,100
1,Bu tür ayarların nasıl yapıldığını bilmiyorum.,3-5 saat,30
2,"Evet, teknik ayarları (filtre, süre sınırı) ak...",1-3 saat,91
3,Nasıl yapılacağını biliyorum ama uygulamıyorum.,1 saatten az,77
4,Nasıl yapılacağını biliyorum ama uygulamıyorum.,1-3 saat,85


In [7]:
display(df_veliler.sample(5, random_state=42))

,veli_cinsiyet,veli_yas,cocuk_cinsiyet,cocuk_yas,cocuk_cihaz,veli_ebeveyn_denetimi,cocuk_oyun_turu,cocuk_gunluk_sure,veli_izin_verilen_harcama,veli_harcama_guven_etkisi,cocuk_kapatma_tepkisi,cocuk_stres_gozlemi,cocuk_uyku_gozlemi,cocuk_ruh_hali_gozlemi,veli_pegi_kontrol,veli_platform_kisitlama_gorus,cocuk_haftalik_oyun_gunu,veli_ebeveyn_onay_gorus,ebeveyn_ilgi_skoru_100
0,Kadın,18-22,Kız,10-13,"Mobil Cihaz (Akıllı Telefon, Tablet)","Evet, teknik ayarları (filtre, süre sınırı) ak...","Basit Eğlence / Bulmaca (Candy Crush, Kelime O...",1 saatten az,Hiç harcama yapmaz,Hiç harcamaz veya sadece izin verdiğimiz kadar...,"""Birazdan"", ""Son bir el"" diyerek süreyi esnetm...","Tamamen rahatlar, neşeli ve stressiz olur.",Hiçbir zaman,Hemen normal hayatına döner.,Evet her zaman,Kararsızım,Neredeyse her gün / Her gün,Kesinlikle destekliyorum,100
13,Erkek,46-55,Erkek,5 yaş ve altı,"Mobil Cihaz (Akıllı Telefon, Tablet)",Bu tür ayarların nasıl yapıldığını bilmiyorum.,"Hayatta Kalma, İnşa Etme & Sandbox (Minecraft,...",3-5 saat,Hiç harcama yapmaz,İlgilenmiyorum / Bilmiyorum,"Kapatmamak için tartışır, sinirlenir, sesini y...","Sadece oynadığı sürece keyiflidir, kafası dağı...",Sıklıkla (Haftada birkaç kez),Çevresine tahammülsüz veya gergin olur.,Bazen,Kararsızım,Sıklıkla (Haftada 3-4 kez),Kesinlikle destekliyorum,23
8,Kadın,29-35,Kız,6-9,"Mobil Cihaz (Akıllı Telefon, Tablet)",Cihazlarda böyle özelliklerin olduğundan haber...,Bilmiyorum,1 saatten az,Hiç harcama yapmaz,Hiç harcamaz veya sadece izin verdiğimiz kadar...,"""Birazdan"", ""Son bir el"" diyerek süreyi esnetm...",Nötr (Oynaması veya oynamaması genel ruh halin...,Hiçbir zaman,"Sadece anlık olarak canı sıkılır, çabuk geçer.",Evet her zaman,Kesinlikle destekliyorum,Neredeyse her gün / Her gün,Kesinlikle destekliyorum,78
1,Kadın,36-45,Erkek,14-17,"Mobil Cihaz (Akıllı Telefon, Tablet)",Bu tür ayarların nasıl yapıldığını bilmiyorum.,"Hayatta Kalma, İnşa Etme & Sandbox (Minecraft,...",3-5 saat,250 TL'den az,Bazen limitini aşar ama bütçemizi sarsmaz.,"""Birazdan"", ""Son bir el"" diyerek süreyi esnetm...",Oynamadığı zamanlarda canı çabuk sıkılır veya ...,Sıklıkla (Haftada birkaç kez),O günkü genel morali düşer.,Hayır kontrol etmiyorum,Destekliyorum,Sıklıkla (Haftada 3-4 kez),Kesinlikle destekliyorum,30
15,Kadın,46-55,Kız,10-13,"Mobil Cihaz (Akıllı Telefon, Tablet)",Nasıl yapılacağını biliyorum ama uygulamıyorum.,"Hayatta Kalma, İnşa Etme & Sandbox (Minecraft,...",1-3 saat,Hiç harcama yapmaz,Hiç harcamaz veya sadece izin verdiğimiz kadar...,"""Birazdan"", ""Son bir el"" diyerek süreyi esnetm...",Nötr (Oynaması veya oynamaması genel ruh halin...,Hiçbir zaman,"Sadece anlık olarak canı sıkılır, çabuk geçer.",PEGI nedir bilmiyorum,Destekliyorum,Nadiren (Ayda 1-2 kez),Kesinlikle destekliyorum,69


In [8]:
print("="*60)
print("💾 SKORLANMIŞ VELİ VERİSİNİ KAYDETME")
print("="*60)

# Veriyi Streamlit'in okuyabileceği şekilde kaydediyoruz
df_veliler.to_csv("../../data/processed/veliler_skorlu.csv", index=False)
print("✅ Veli verisi başarıyla kaydedildi! Streamlit'e geçişe hazırız.")

💾 SKORLANMIŞ VELİ VERİSİNİ KAYDETME
✅ Veli verisi başarıyla kaydedildi! Streamlit'e geçişe hazırız.
